In [0]:
# STARTER CODE - DO NOT EDIT THIS CELL

from pyspark.sql.functions import*
from pyspark.sql.types import *
from pyspark.sql import Window
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


In [0]:
# STARTER CODE - DO NOT EDIT THIS CELL

#from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

customSchema = StructType([
    StructField("lpep_pickup_datetime", StringType(), True),
    StructField("lpep_dropoff_datetime", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("payment_type", IntegerType(), True)
])

In [0]:
# STARTER CODE - YOU CAN LOAD ANY FILE WITH A SIMILAR SYNTAX.
# Correct file path for Databricks File System (update if you are using a different volume or file name)
file_path = "/Volumes/workspace/default/q2vol/nyc-tripdata.csv"

# Read the CSV file using Spark DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(df.limit(5))


In [0]:
# LOAD THE "taxi_zone_lookup.csv" FILE SIMILARLY AS ABOVE. CAST ANY COLUMN TO APPROPRIATE DATA TYPE IF NECESSARY.

#ENTER THE CODE BELOW
file_path = "/Volumes/workspace/default/q2vol/taxi_zone_lookup.csv"

# Read the CSV file using Spark DataFrame
taxi_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(taxi_df.limit(5))


In [0]:
#// STARTER CODE 
# // Some commands that you can use to see your dataframes and results of the operations. You can comment the df.show(5) and uncomment display(df) to see the data differently. You will find these two functions useful in reporting your results.
df.show(5)
#display(df.limit(5))

In [0]:
# // STARTER CODE - DO NOT EDIT THIS CELL
# Filter the data to only keep the rows where "PULocationID" and the "DOLocationID" are different and the "trip_distance" is strictly greater than 2.0 (>2.0).

# VERY VERY IMPORTANT: ALL THE SUBSEQUENT OPERATIONS MUST BE PERFORMED ON THIS FILTERED DATA

df_filter = df.filter((df.PULocationID != df.DOLocationID) & (df.trip_distance > 2.0))
df_filter.show(5)

In [0]:
# PART 1a: The top-5 most popular drop locations - "DOLocationID", sorted in descending order - if there is a tie, then one with lower "DOLocationID" gets listed first

# Output Schema: DOLocationID int, number_of_dropoffs int 

# Hint: Checkout the groupBy(), orderBy() and count() functions.

# ENTER THE CODE BELOW

res = df_filter .groupBy("DOLocationID") .agg(count("*").cast("int").alias("num_dropoffs")).orderBy(desc("num_dropoffs"), asc("DOLocationID")) .limit(5)
res.show()




In [0]:
# PART 1b: The top-5 most popular pickup locations - "PULocationID", sorted in descending order - if there is a tie, then one with lower "PULocationID" gets listed first 

# Output Schema: PULocationID int, number_of_pickups int

# ENTER THE CODE BELOW

res = df_filter.groupBy("PULocationID") .agg(count("*").cast("int").alias("num_pickups")).orderBy(desc("num_pickups"), asc("PULocationID")) .limit(5)
res.show()




In [0]:
# PART 2: List the top-3 locations with the maximum overall activity, i.e. sum of all pickups and all dropoffs at that LocationID. In case of a tie, the lower LocationID gets listed first.

# Output Schema: LocationID int, number_activities int

# Hint: In order to get the result, you may need to perform a join operation between the two dataframes that you created in earlier parts (to come up with the sum of the number of pickups and dropoffs on each location). 

# ENTER THE CODE BELOW
pickup_count = df_filter.groupBy("PULocationID").agg(count("*").alias("num_pickups")).withColumnRenamed("PULocationID", "LocationID")
dropoff_count = df_filter.groupBy("DOLocationID").agg(count("*").alias("num_dropoffs")).withColumnRenamed("DOLocationID", "LocationID")
temp = pickup_count.join(dropoff_count, "LocationID").withColumn("num_activities", pickup_count.num_pickups + dropoff_count.num_dropoffs).orderBy(desc("num_activities"), asc("LocationID"))
res = temp.limit(3)

res = res.drop("num_pickups", "num_dropoffs") 
res.show()


In [0]:
# PART 3: List all the boroughs (including "Unknown" and "EWR") in the order of having the highest to lowest number of activities (i.e. sum of all pickups and all dropoffs at that LocationID), along with the total number of activity counts for each borough in NYC during that entire period of time.

# Output Schema: Borough string, total_number_activities int

# Hint: You can use the dataframe obtained from the previous part, and will need to do the join with the 'taxi_zone_lookup' dataframe. Also, checkout the "agg" function applied to a grouped dataframe.

# ENTER THE CODE BELOW

#i had to load it again idk why the prev one wont work
taxi_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/q2vol/taxi_zone_lookup.csv")

temp = temp.drop("num_pickups", "num_dropoffs")
result = temp.join(taxi_df, "LocationID").groupBy("Borough").agg(sum("num_activities").alias("total_number_activities")).orderBy(desc("total_number_activities"))

result.show()
display(result)


In [0]:
# PART 4: List the top 2 days of week with the largest number of daily average pickups, along with the average number of pickups on each of the 2 days in descending order (no rounding off required). Here, the average pickup is calculated by taking an average of the number of pick-ups on different dates falling on the same day of the week. For example, 02/01/2021, 02/08/2021 and 02/15/2021 are all Mondays, so the average pick-ups for these is the sum of the pickups on each date divided by 3.

# Note: The day of week is a string of the day’s full spelling, e.g., "Monday" instead of the		number 1 or "Mon". Also, the pickup_datetime is in the format: yyyy-mm-dd.

# Output Schema: day_of_week string, avg_count float

# Hint: You may need to group by the "date" (without time stamp - time in the day) first. Checkout "date_format" and "to_date" functions.

# ENTER THE CODE BELOW

df_date = df_filter.select(to_date(col("lpep_pickup_datetime"), "MM/dd/yyyy").alias("date"))

counts = df_date.groupBy("date").agg(count("*").alias("daily_pickups"))

res = counts.withColumn("day_of_week", date_format(col("date"), "EEEE")).groupBy("day_of_week").agg(avg("daily_pickups").alias("avg_count")).orderBy(desc("avg_count")).limit(2)

display(res)




In [0]:
# PART 5: For each particular hour of a day (0 to 23, 0 being midnight) - in their order from 0 to 23, find the zone in Brooklyn borough with the LARGEST number of pickups. 

# Note: All dates for each hour should be included.

# Output Schema: hour_of_day int, zone string, max_count int

# Hint: You may need to use "Window" over hour of day, along with "group by" to find the MAXIMUM count of pickups

# ENTER THE CODE BELOW
taxi_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/q2vol/taxi_zone_lookup.csv")

res = taxi_df.alias("t").join(
    df_filter.alias("d"),
    col("t.LocationID") == col("d.PULocationID")
)


res = res.filter(col("t.Borough") == "Brooklyn").withColumn("hour_of_day", hour(to_timestamp(col("d.lpep_pickup_datetime"), "MM/dd/yyyy HH:mm")))
res = res.groupBy("hour_of_day", "t.Zone").agg(count("*").alias("total_pickups"))
window = Window.partitionBy("hour_of_day").orderBy(desc("total_pickups"))
res = res.withColumn( "rank", row_number().over(window) )
res = res.filter(col("rank") == 1).select("hour_of_day", "t.Zone", "total_pickups").orderBy("hour_of_day")

res_clean = res.select(
    col("hour_of_day"),
    col("Zone").alias("zone"),
    col("total_pickups").alias("max_count")
)

print("hour_of_day,zone,max_count")

for row in res_clean.collect():
    print(f"{row['hour_of_day']},{row['zone']},{row['max_count']}")




In [0]:
# PART 6 - Find which 3 different days in the month of January, in Manhattan, saw the largest positive percentage increase in pick-ups compared to the previous day, in the order from largest percentage increase to smallest percentage increase 

# Note: All years need to be aggregated to calculate the pickups for a specific day of January. The change from Dec 31 to Jan 1 can be excluded.

# Output Schema: day int, percent_change float

# Hint: You might need to use lag function, over a window ordered by day of month.

# ENTER THE CODE BELOW
taxi_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/q2vol/taxi_zone_lookup.csv")
location = taxi_df.filter(col("Borough") == "Manhattan").select("LocationID")

res = df_filter.join(location, df_filter.PULocationID == location.LocationID)
res = res.filter(month(to_date(col("lpep_pickup_datetime"), "MM/dd/yyyy H:mm")) == 1).withColumn("day", dayofmonth(to_date(col("lpep_pickup_datetime"), "MM/dd/yyyy H:mm")))

agg = [count("*").alias("daily_pickups")]
counts = res.groupBy("day").agg(*agg)

window = Window.orderBy("day")
counts = counts.withColumn("prev", lag("daily_pickups").over(window))
counts = counts.filter(col("prev").isNotNull())
counts = counts.withColumn("percent_change", ((col("daily_pickups") - col("prev")) / col("prev")) * 100)
rows = counts.orderBy(desc("percent_change")).select("day", "percent_change").limit(3).collect()

for row in rows:
    print(row["day"], row["percent_change"])